# Annotation Quality Audit Program

This program audits the annotation quality of dogwhistle terms in the benchmark dataset.
It decomposes annotations into cases A (present + hateful), B (present + non-hateful),
and C (absent from union), and calculates labeling accuracy metrics.

In [ ]:
# Imports
from dataclasses import dataclass
from pathlib import Path

import pandas as pd


In [ ]:
@dataclass
class AnnotationQualityAuditConfig:
    # `workdir` is the anchor used to resolve all relative paths.
    workdir: Path

    # Local sources expected to exist in the repository.
    # Stage-06 glossary-mapped cleaned labels are the default audit input.
    data_local_path: Path = Path('../outputs/unioned_data/06_cleaned_labels_glossary_mapped.tsv')
    coverage_output_dir: Path = Path('../outputs/coverage_audits')

    # Directory where audit exports are written.
    output_dir: Path = Path('../outputs/annotation_audits')

In [ ]:
# Configuration
WORKDIR = Path.cwd()
cfg = AnnotationQualityAuditConfig(workdir=WORKDIR)

# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.coverage_output_path = cfg.workdir / cfg.coverage_output_dir
cfg.upstream_matches_path = cfg.coverage_output_path / 'audit_matches.tsv'
cfg.upstream_metrics_path = cfg.coverage_output_path / 'audit_metrics.tsv'

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

cfg


In [ ]:
# Load data and stage-00 pipeline outputs
data = pd.read_csv(cfg.data_path, sep='\t')
audit_df = pd.read_csv(cfg.upstream_matches_path, sep='\t')
coverage_report = pd.read_csv(cfg.upstream_metrics_path, sep='\t')

# Ensure everything is lowercase for matching and joins
data['targets'] = data['targets'].fillna('').str.lower()
audit_df['target'] = audit_df['target'].astype(str).str.strip().str.lower()
audit_df['dogwhistle'] = audit_df['dogwhistle'].astype(str).str.strip().str.lower()
audit_df['found_forms'] = audit_df['found_forms'].astype(str).str.strip().str.lower()

print(f"Loaded stage-00 matches: {len(audit_df)} rows")
print(f"Loaded stage-00 coverage metrics: {len(coverage_report)} rows")


In [ ]:
# Validate required upstream columns
required_cols = {
    'text_dedup_key', 'binary_hate', 'found_forms', 'dogwhistle',
    'taxonomy_level', 'target', 'type'
}
missing_cols = required_cols - set(audit_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns in stage-00 output: {sorted(missing_cols)}")

audit_df.head()

In [ ]:
# Build annotation quality metrics
annotation_quality_data = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    # Equal weighting: one count per (post, dogwhistle) regardless of surface-form variants.
    dogwhistle_post_hits = group.drop_duplicates(['text_dedup_key', 'dogwhistle'])

    # Case A: Dogwhistle hit labeled hateful
    case_a = len(dogwhistle_post_hits[dogwhistle_post_hits['binary_hate'] == 1])

    # Case B: Dogwhistle hit labeled non-hateful
    case_b = len(dogwhistle_post_hits[dogwhistle_post_hits['binary_hate'] == 0])

    # Case C: Target-group posts with no dogwhistle hit for this level/target
    target_group_posts = data[data['targets'].str.contains(target, case=False, na=False)]
    posts_with_dogwhistles = dogwhistle_post_hits['text_dedup_key'].unique()
    case_c = len(target_group_posts[~target_group_posts['text_dedup_key'].isin(posts_with_dogwhistles)])

    matches_with_label = case_a + case_b
    correct_labeling_rate = case_a / matches_with_label if matches_with_label > 0 else 0
    annotator_failure_ratio = case_b / matches_with_label if matches_with_label > 0 else 0

    annotation_quality_data.append({
        'taxonomy_level': level,
        'target': target,
        'case_a_present_hateful': case_a,
        'case_b_present_nonhateful': case_b,
        'case_c_absent': case_c,
        'correct_labeling_rate': correct_labeling_rate,
        'annotator_failure_ratio': annotator_failure_ratio,
        'total_matches': matches_with_label,
        'total_target_group_posts': len(target_group_posts)
    })

annotation_quality_report = pd.DataFrame(annotation_quality_data)
annotation_quality_report

In [ ]:
# Analyze case breakdown: B/(B+C) indicates whether gap is annotation vs. collection
case_breakdown = annotation_quality_report.copy()
case_breakdown['case_b_c_ratio'] = (
    case_breakdown['case_b_present_nonhateful'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)
case_breakdown['collection_gap_prop'] = (
    case_breakdown['case_c_absent'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)

print("=" * 100)
print("ANNOTATION QUALITY AUDIT: CASE BREAKDOWN")
print("=" * 100)
print("\nCase B/C Ratio: Case B / (Case B + Case C)")
print("  - Closer to 1.0: primarily annotation failure (forms are there, missed by annotators)")
print("  - Closer to 0.0: primarily collection failure (forms not in benchmark at all)")
print("\n")
print(case_breakdown[['taxonomy_level', 'target', 'case_a_present_hateful', 
                       'case_b_present_nonhateful', 'case_c_absent', 
                       'case_b_c_ratio', 'collection_gap_prop']].to_string(index=False))

print("\n" + "=" * 100)
print("INTERPRETATION SUMMARY")
print("=" * 100)
for idx, row in case_breakdown.iterrows():
    level = row['taxonomy_level']
    target = row['target']
    ratio = row['case_b_c_ratio']
    b = row['case_b_present_nonhateful']
    c = row['case_c_absent']
    
    if b + c > 0:
        if ratio > 0.7:
            problem = "ANNOTATION FAILURE (primary issue is mislabeling)"
        elif ratio < 0.3:
            problem = "COLLECTION FAILURE (primary issue is missing forms)"
        else:
            problem = "MIXED (both annotation and collection issues)"
    else:
        problem = "INSUFFICIENT DATA"
    
    print(f"\nLevel {level}, {target.upper()}: {problem}")
    print(f"  Case A (correct): {row['case_a_present_hateful']} | " +
          f"Case B (missed): {b} | Case C (absent): {c}")

In [ ]:
# Detailed breakdown by dogwhistle: which dogwhistles are mislabeled (case B)
form_labeling_detail = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    for dogwhistle, dogwhistle_group in group.groupby('dogwhistle'):
        dedup_hits = dogwhistle_group.drop_duplicates(['text_dedup_key', 'dogwhistle'])

        hateful_count = len(dedup_hits[dedup_hits['binary_hate'] == 1])
        nonhateful_count = len(dedup_hits[dedup_hits['binary_hate'] == 0])

        labeling_accuracy = (
            hateful_count / (hateful_count + nonhateful_count)
            if (hateful_count + nonhateful_count) > 0 else 0
        )

        matched_forms = '; '.join(sorted(dogwhistle_group['found_forms'].dropna().unique()))

        form_labeling_detail.append({
            'taxonomy_level': level,
            'target': target,
            'dogwhistle': dogwhistle,
            'matched_surface_forms': matched_forms,
            'case_a_count': hateful_count,
            'case_b_count': nonhateful_count,
            'labeling_accuracy': labeling_accuracy,
            'type': dogwhistle_group['type'].iloc[0] if len(dogwhistle_group) > 0 else 'unknown'
        })

form_labeling_report = pd.DataFrame(form_labeling_detail).sort_values(
    by=['taxonomy_level', 'target', 'case_b_count'], ascending=[True, True, False]
 )

print("\n" + "=" * 100)
print("DOGWHISTLE LABELING DETAILS (Case A vs Case B)")
print("=" * 100)
print("\nDogwhistles sorted by mislabeling frequency (Case B count):")
print(form_labeling_report.to_string(index=False))

In [ ]:
# Export annotation quality audit results
annotation_quality_path = cfg.output_dir / 'annotation_quality.tsv'
case_breakdown_path = cfg.output_dir / 'case_breakdown.tsv'
form_labeling_path = cfg.output_dir / 'form_labeling_detail.tsv'
coverage_passthrough_path = cfg.output_dir / 'coverage_metrics_upstream.tsv'

annotation_quality_report.to_csv(annotation_quality_path, sep='\t', index=False)
case_breakdown.to_csv(case_breakdown_path, sep='\t', index=False)
form_labeling_report.to_csv(form_labeling_path, sep='\t', index=False)

# Stage-02 pipeline input: pass through stage-00 coverage metrics.
coverage_report.to_csv(coverage_passthrough_path, sep='\t', index=False)

print("\nAnnotation quality audit exported:")
print(f"  - {annotation_quality_path}")
print(f"  - {case_breakdown_path}")
print(f"  - {form_labeling_path}")
print(f"  - {coverage_passthrough_path}")